##Notebook for assessing how corrected data affects model performance

In [1]:
import os
import sys
import logging

# Silence Hugging Face internet retry warnings in offline environments.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.file_download").setLevel(logging.ERROR)

try:
    from huggingface_hub import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

sys.path.insert(0, r"D:\Politechnika\Deadlift_publikacja_nr_2 — kopia\Skrypty\github")


from utils.datasets import YOLOPoseDataset, save_results_console_output, merge_yolo_pose_datasets,split_yolo_pose_dataset
from utils.validation import YPSetValidation, MergedRankingEvaluator

c:\Users\rwcki\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##Datasets making

In [6]:

dataset_full = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged\train.txt")
validator = YPSetValidation(dataset_full)
embs,filenames=validator.get_embeddings()
duplicates=validator.find_near_duplicates(embs,filenames,100)
result = dataset_full.remove_duplicates(duplicates, keep="first",output_dataset_path = r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged_bez_duplikatow")


dataset_full = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\Iteracja_5_merged_bez_duplikatow\train.txt")
####Podział na 3 części po usunieciu duplikatow 
dataset_full.export_dataset_partitions(3, r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions")
dataset_1 = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1\train.txt")
dataset_2 = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2\train.txt")
dataset_3 = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3\train.txt")

#part_1 - bez zmian part, z dopiskiem zmienione
dataset_path_1 = r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1"
dataset_path_2 = r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2"
dataset_path_3 = r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3"
x=150
max_shift=0.25
min_shift =0.05

result = dataset_1.replace_angle_preserving_keypoints(
    sample_count=x,
    max_keypoints_to_shift=3,
    max_endpoint_shift=max_shift,
    min_shift = min_shift,
    output_dataset_path=dataset_path_1 + f"_angle_preserving_out",
)
save_results_console_output(result, os.path.join(dataset_path_1 + f"_angle_preserving_out","results.txt"), include_details=True,details_one_line=True)

results = dataset_2.replace_rigid_keypoints(
    sample_count=x,
    max_rotation_degrees=25.0,
    max_translation_distance=0.25,
    transform_mode='both',
    output_dataset_path=dataset_path_2 + f"_rigid_out",
)
save_results_console_output(results, os.path.join(dataset_path_2 + f"_rigid_out","results.txt"), include_details=True,details_one_line=True)

results = dataset_3.replace_axis_aligned_keypoints(
    sample_count=x,
    max_keypoints_to_shift=3,
    max_shift_distance=0.25,
    direction_mode='both',
    output_dataset_path=dataset_path_3 + f"_axis_out",
    seed=123,
 )
save_results_console_output(results, os.path.join(dataset_path_3 + f"_axis_out","results.txt"), include_details=True,details_one_line=True)

merge_yolo_pose_datasets(os.path.join(dataset_path_3 + f"_axis_out"), os.path.join(dataset_path_2 + f"_rigid_out"), r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset")
merge_yolo_pose_datasets(os.path.join(dataset_path_1 + f"_angle_preserving_out"),r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset", r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset_brudny")

dataset_1_czysty=YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1\train.txt")
dataset_2_czysty = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2\train.txt")
dataset_3_czysty = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3\train.txt")

merge_yolo_pose_datasets(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1",r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2",r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1_2_merged")
merge_yolo_pose_datasets(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1_2_merged",r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3",r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\full_czysty_dataset")



Computing embeddings: 100%|██████████| 1797/1797 [00:59<00:00, 30.26it/s]


## Quick checking

In [32]:
dataset_czysty_bez_duplikatow = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\full_czysty_dataset\train.txt")

In [33]:
validator=YPSetValidation(dataset_czysty_bez_duplikatow)
validator.get_stats()

{'total_images': 1793,
 'total_persons': 1919,
 'total_keypoints': 17271,
 'total_visible_keypoints': 16627,
 'avg_keypoints_per_image': 9.632459564974903,
 'avg_keypoints_per_person': 9.0,
 'avg_visible_keypoints_per_image': 9.273284997211377,
 'avg_visible_keypoints_per_person': 8.66440854611777,
 'dataset_file': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\full_czysty_dataset\\train.txt',
 'labels_dir': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\full_czysty_dataset\\labels'}

In [10]:
dataset_brudny = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset_brudny\train.txt")
validator = YPSetValidation(dataset_brudny)

In [11]:
validator.get_stats()

{'total_images': 1793,
 'total_persons': 1919,
 'total_keypoints': 17271,
 'total_visible_keypoints': 16627,
 'avg_keypoints_per_image': 9.632459564974903,
 'avg_keypoints_per_person': 9.0,
 'avg_visible_keypoints_per_image': 9.273284997211377,
 'avg_visible_keypoints_per_person': 8.66440854611777,
 'dataset_file': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\final_augmented_dataset_brudny\\train.txt',
 'labels_dir': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\final_augmented_dataset_brudny\\labels'}

## Merging text files

In [9]:
def merge_three_text_files(input_path_1, input_path_2, input_path_3, output_path, add_newline_between_files=True):
    input_paths = [input_path_1, input_path_2, input_path_3]
    merged_chunks = []

    for input_path in input_paths:
        with open(input_path, "r", encoding="utf-8") as file_handle:
            content = file_handle.read().rstrip()
        merged_chunks.append(content)

    separator = "\n\n" if add_newline_between_files else "\n"
    merged_content = separator.join(chunk for chunk in merged_chunks if chunk)

    with open(output_path, "w", encoding="utf-8") as file_handle:
        file_handle.write(merged_content + ("\n" if merged_content else ""))

    return output_path

output_txt_path = merge_three_text_files(
    input_path_1=r"D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\partitions\\part_1_angle_preserving_out\\results.txt",
    input_path_2=r"D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\partitions\\part_2_rigid_out\\results.txt",
    input_path_3=r"D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\partitions\\part_3_axis_out\\results.txt",
    output_path=r"D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\partitions\\merged_results.txt",
    add_newline_between_files=False,
 )

print("Zapisano:", output_txt_path)

Zapisano: D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\partitions\\merged_results.txt


## Validation using our method

In [2]:
from utils.validation import YPValidation_Test, parse_results_txt


def test(
    x,
    distance_weight=0.5,
    lbp_weight=0.1,
    segmentation_weight=0.4,
    output_path=None,
):
    dataset = YOLOPoseDataset(
        r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset_brudny\train.txt"
    )
    validator = YPSetValidation(dataset)

    report_paths = {
        "angle_preserving": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1_angle_preserving_out\results.txt",
        "rigid": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2_rigid_out\results.txt",
        "axis_aligned": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3_axis_out\results.txt",
    }

    evaluator = MergedRankingEvaluator(
        validator=validator,
        report_paths=report_paths,
        top_k=x,
        output_path=None,
        distance_model_name="random_forest",
        distance_visibility_threshold=2.0,
        distance_threshold_percentile=95.0,
        distance_sort_by="mean",
        segmentation_mask_tolerance_px=24,
        segmentation_model_name="yolo26l-seg",
        segmentation_target_class="person"
    )

    result = evaluator.run(
        distance_weight=distance_weight,
        lbp_weight=lbp_weight,
        segmentation_weight=segmentation_weight,
    )

    if not result:
        return None

    # Zwracamy pelny wynik, bo jest potrzebny do odtworzenia listy brakujacych rekordow.
    result["top_k"] = x
    return result

In [3]:
import time

cell_start = time.perf_counter()
merged_450 = test(
    x=450,
    distance_weight=0.65,
    lbp_weight=0.1,
    segmentation_weight=0.25
)
cell_elapsed_seconds = time.perf_counter() - cell_start

print(f"Czas wykonania komórki: {cell_elapsed_seconds:.2f} s")
merged_450

Extracting sequential distances (visibility_threshold=2.0): 100%|██████████| 1793/1793 [00:16<00:00, 110.24it/s]
Extracting sequential distances (visibility_threshold=2.0): 100%|██████████| 1793/1793 [00:16<00:00, 111.61it/s]
c:\Users\rwcki\anaconda3\envs\torch_env\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
Computing embeddings: 100%|██████████| 1793/1793 [00:59<00:00, 30.19it/s]


All similarity groups: 94
Training groups with size > 10: 12


Evaluating keypoints vs masks: 100%|██████████| 1793/1793 [00:18<00:00, 95.88it/s] 


weights=0.650/0.100/0.250 | merged=320/450 | overall=450/450 | avg_rank_all=431.37 | angle_preserving=106/150 | overall=150/150 | avg_rank_all=549.97 | rigid=87/150 | overall=150/150 | avg_rank_all=501.65 | axis_aligned=127/150 | overall=150/150 | avg_rank_all=242.49
CACHE READY | all_samples=1919 | distance=1615 | lbp=235 | mask_ranked_records=1919 | segmentation_with_mask=1786 | segmentation_without_mask=7 | outside=175 | outside_person_total=175 | outside_persons_in_top_k=175 | outside_mentioned_in_merged=0
Czas wykonania komórki: 378.56 s


{'weights': (0.65, 0.1, 0.25),
 'output_path': None,
 'ranking_meta': {'promoted_outside_count': 175,
  'ranked_after_outside_count': 1919,
  'outside_total': 175,
  'outside_person_total': 175,
  'outside_in_3_models': 175,
  'outside_mentioned_in_merged': 0,
  'segmentation_model_name': 'yolo26l-seg',
  'segmentation_target_class_name': 'person',
  'segmentation_target_class_id': 0,
  'segmentation_mask_tolerance_px': 24,
  'mask_ranked_records_total': 1919,
  'segmentation_total_images': 1793,
  'segmentation_images_with_mask': 1786,
  'segmentation_images_without_mask': 7,
  'outside_persons_in_top_k': 175},
 'combined_ranking': [{'image_path': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\final_augmented_dataset_brudny\\images/Conventional-Deadlift-side-view_frame_5_jpg.rf.89e294a4a04d01858e6ed7020894dbaa.jpg',
   'person_id': 0,
   'outside_mask_detected': True,
   'outside_keypoint_count': 2,
   'distance_powered_score': 2515396.0,
   'lbp_powered_score': 30976.

## Reference baseline using random lists

In [21]:
import random
import statistics

from utils.validation import parse_results_txt, YPValidation_Test

N_RUNS = 100
TOP_K = 450
SEED = 123

# Referencja (to, co chcemy znaleźć w top_k)
merged_reference_results_path = r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\merged_results.txt"
reference_entries = parse_results_txt(merged_reference_results_path)
reference_total = len(reference_entries)

if reference_total == 0:
    raise ValueError("Brak rekordów referencyjnych w merged_results.txt")

# Budujemy pulę rekordów WYŁĄCZNIE z dataset_brudny, bez modelu i bez dystansów.
# Każda osoba (linia w labelu) to osobny rekord: (image_path, person_id).
all_random_candidates = []
for image_path, label_path in getattr(dataset_brudny, "_valid_pairs", []):
    try:
        with open(label_path, "r", encoding="utf-8") as file_handle:
            label_lines = [line.strip() for line in file_handle.readlines() if line.strip()]
    except OSError:
        continue

    for person_id, _ in enumerate(label_lines):
        all_random_candidates.append(
            {
                "image_path": str(image_path),
                "person_id": int(person_id),
            }
        )

if not all_random_candidates:
    raise ValueError("Brak rekordów do losowania w dataset_brudny")


def _identifier(entry):
    return YPValidation_Test._distance_anomaly_entry_identifier(entry, "image_path_person_id")


reference_identifiers = [_identifier(entry) for entry in reference_entries]
rng = random.Random(SEED)
run_details = []

for run_idx in range(1, N_RUNS + 1):
    shuffled_candidates = all_random_candidates[:]
    rng.shuffle(shuffled_candidates)

    top_entries = shuffled_candidates[:TOP_K]

    top_buckets = {}
    for rank, entry in enumerate(top_entries, start=1):
        identifier = _identifier(entry)
        top_buckets.setdefault(identifier, []).append(rank)

    matched_top_positions = []
    for reference_identifier in reference_identifiers:
        bucket = top_buckets.get(reference_identifier, [])
        if bucket:
            matched_top_positions.append(bucket.pop(0))

    matched_top_count = len(matched_top_positions)
    avg_top_position = (
        float(sum(matched_top_positions) / matched_top_count)
        if matched_top_count > 0
        else None
    )

    run_details.append(
        {
            "run": run_idx,
            "matched_top_k_count": matched_top_count,
            "reference_total": reference_total,
            "hit_rate": float(matched_top_count / reference_total),
            "avg_top_position": avg_top_position,
        }
    )

matched_counts = [row["matched_top_k_count"] for row in run_details]
avg_positions = [row["avg_top_position"] for row in run_details if row["avg_top_position"] is not None]
hit_rates = [row["hit_rate"] for row in run_details]

random_baseline_summary = {
    "runs": N_RUNS,
    "top_k": TOP_K,
    "reference_total": reference_total,
    "candidate_pool_size": len(all_random_candidates),
    "mean_matched_top_k_count": float(statistics.mean(matched_counts)) if matched_counts else 0.0,
    "std_matched_top_k_count": float(statistics.pstdev(matched_counts)) if len(matched_counts) > 1 else 0.0,
    "mean_hit_rate": float(statistics.mean(hit_rates)) if hit_rates else 0.0,
    "mean_avg_top_position": float(statistics.mean(avg_positions)) if avg_positions else None,
}

print("Random-only baseline summary:")
for key, value in random_baseline_summary.items():
    print(f"  {key}: {value}")

run_details[:10]

Random-only baseline summary:
  runs: 100
  top_k: 450
  reference_total: 450
  candidate_pool_size: 1919
  mean_matched_top_k_count: 105.18
  std_matched_top_k_count: 7.332639361103204
  mean_hit_rate: 0.23373333333333332
  mean_avg_top_position: 227.30112849473997


[{'run': 1,
  'matched_top_k_count': 102,
  'reference_total': 450,
  'hit_rate': 0.22666666666666666,
  'avg_top_position': 231.61764705882354},
 {'run': 2,
  'matched_top_k_count': 114,
  'reference_total': 450,
  'hit_rate': 0.25333333333333335,
  'avg_top_position': 229.57894736842104},
 {'run': 3,
  'matched_top_k_count': 107,
  'reference_total': 450,
  'hit_rate': 0.23777777777777778,
  'avg_top_position': 234.15887850467288},
 {'run': 4,
  'matched_top_k_count': 96,
  'reference_total': 450,
  'hit_rate': 0.21333333333333335,
  'avg_top_position': 225.07291666666666},
 {'run': 5,
  'matched_top_k_count': 102,
  'reference_total': 450,
  'hit_rate': 0.22666666666666666,
  'avg_top_position': 225.62745098039215},
 {'run': 6,
  'matched_top_k_count': 94,
  'reference_total': 450,
  'hit_rate': 0.2088888888888889,
  'avg_top_position': 237.60638297872342},
 {'run': 7,
  'matched_top_k_count': 101,
  'reference_total': 450,
  'hit_rate': 0.22444444444444445,
  'avg_top_position': 22

## Validation using MITL approach

In [4]:
dataset = YOLOPoseDataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset_brudny\train.txt")
validator = YPSetValidation(dataset)
def test_mitl(
    x,
    model,
    initial_epochs,
    remove_percent,
    fine_tune_epochs,
    train_kwargs=None,
    fine_tune_train_kwargs=None,
    predict_kwargs=None,
    lightweight_return=True,
    lightweight_top_k=None,
):
    import os
    import shutil



    # Compatibility fix for datasets with images/train and flat labels/*.txt
    dataset_dir = os.path.abspath(getattr(dataset, "dataset_dir", ""))
    labels_root = os.path.join(dataset_dir, "labels")
    labels_train = os.path.join(labels_root, "train")
    if os.path.isdir(labels_root):
        os.makedirs(labels_train, exist_ok=True)
        for image_path, _ in getattr(dataset, "_valid_pairs", []):
            label_name = os.path.splitext(os.path.basename(str(image_path)))[0] + ".txt"
            src = os.path.join(labels_root, label_name)
            dst = os.path.join(labels_train, label_name)
            if not os.path.isfile(src) or os.path.exists(dst):
                continue
            try:
                os.link(src, dst)
            except OSError:
                shutil.copy2(src, dst)

    mitl_result = validator.model_in_the_loop_validation(
        model=model,
        initial_epochs=initial_epochs,
        remove_percent=remove_percent,
        fine_tune_epochs=fine_tune_epochs,
        train_kwargs=train_kwargs,
        fine_tune_train_kwargs=fine_tune_train_kwargs,
        predict_kwargs=predict_kwargs,
        visibility_threshold=0.0,
        lightweight_return=lightweight_return,
        lightweight_top_k=lightweight_top_k,
        verbose=True,
    )

    return mitl_result

In [5]:
# MITL (Model-In-The-Loop) - wywolanie test_mitl
cell_start = time.perf_counter()
mitl_result = test_mitl(
    x=450,
    model="yolo11n-pose.pt",
    initial_epochs=10,
    remove_percent=20,
    fine_tune_epochs=10,
    train_kwargs={"imgsz": 640, "batch": 16, "workers": 0},
    predict_kwargs={"conf": 0.25},
    lightweight_return=False
)
cell_elapsed_seconds = time.perf_counter() - cell_start

print(f"Czas wykonania komórki: {cell_elapsed_seconds:.2f} s")
mitl_result["statistics"] if mitl_result is not None else None

Initial training: 10 epochs on 1793 images
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\rwcki\AppData\Local\Temp\model_in_loop_vggh2zxx\data_full.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

c:\Users\rwcki\anaconda3\envs\torch_env\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/07/21 09:37:21 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/07/21 09:37:21 WARNING mlflow.utils.autologging_utils: MLflow transformers autologging is known to be compatible with 4.40.2 <= transformers, but the installed version is 4.39.3. If you encounter errors during autologging, try upgrading / downgrading transformers to a compatible version, or try upgrading MLflow.
2026/07/21 09:37:22 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 servi

MLflow: logging run_id(b168d914fd504d37baede32ce3c8694b) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\Politechnika\Deadlift_publikacja_nr_2  kopia\Skrypty\github\scores_from_publication\runs\pose\train3
Starting training for 10 epochs...
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
       1/10      2.21G      1.981      7.806     0.6986      2.509      1.839          1        640: 100% ━━━━━━━━━━━━ 113/113 1.5it/s 1:180.7ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Insta

Predicting keypoints (after_initial_training): 100%|██████████| 1793/1793 [01:12<00:00, 24.76it/s]

Continuing training: 10 epochs on 1434 images (359 removed)
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\rwcki\AppData\Local\Temp\model_in_loop_vggh2zxx\data_reduced.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, mu

...

2026/07/21 09:49:47 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/07/21 09:49:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/07/21 09:49:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(2469a094b61a4c039103083ba125f37b) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\Politechnika\Deadlift_publikacja_nr_2  kopia\Skrypty\github\scores_from_publication\runs\pose\train4
Starting training for 10 epochs...
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
       1/10      2.31G      1.563      2.459     0.1887      1.605      1.453         11        640: 100% ━━━━━━━━━━━━ 90/90 1.6it/s 55.3s0.7ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instan

Predicting keypoints (after_reduced_training): 100%|██████████| 1793/1793 [01:12<00:00, 24.62it/s]

Czas wykonania komórki: 1341.59 s


{'initial_epochs': 10,
 'fine_tune_epochs': 10,
 'remove_percent': 20.0,
 'initial_train_image_count': 1793,
 'reduced_train_image_count': 1434,
 'removed_image_count': 359,
 'removed_record_count': 359,
 'final_total_records': 1919,
 'work_dir': 'C:\\Users\\rwcki\\AppData\\Local\\Temp\\model_in_loop_vggh2zxx'}

In [6]:
# Porownanie wynikow MITL z trzema plikami referencyjnymi (ile wykrylismy z tych, ktore powinnismy)
reference_report_paths = {
    "angle_preserving": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1_angle_preserving_out\results.txt",
    "rigid": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2_rigid_out\results.txt",
    "axis_aligned": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3_axis_out\results.txt",
}

mitl_evaluator = MergedRankingEvaluator(
    validator=validator,
    report_paths=reference_report_paths,
    top_k=450,
    output_path=None,
)

reference_entries_map = mitl_evaluator._load_reference_report_entries()
mitl_comparisons = {}
for report_name, reference_entries in reference_entries_map.items():
    mitl_comparisons[report_name] = mitl_evaluator._compare_reference_entries_to_anomalies(
        reference_entries,
        mitl_result["combined_ranking"],
    )

mitl_comparisons

{'angle_preserving': {'matched_top_k_count': 38,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 608.1866666666666,
  'last_matched_rank_in_top_k': 448},
 'rigid': {'matched_top_k_count': 141,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 223.38666666666666,
  'last_matched_rank_in_top_k': 446},
 'axis_aligned': {'matched_top_k_count': 133,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 323.94,
  'last_matched_rank_in_top_k': 442},
 'merged': {'matched_top_k_count': 312,
  'total_reference_entries': 450,
  'matched_overall_count': 450,
  'avg_rank_all_reference': 385.1711111111111,
  'last_matched_rank_in_top_k': 448}}

In [7]:
# MITL (Model-In-The-Loop) - wywolanie test_mitl
cell_start = time.perf_counter()
mitl_result = test_mitl(
    x=450,
    model="yolo11n-pose.pt",
    initial_epochs=20,
    remove_percent=20,
    fine_tune_epochs=20,
    train_kwargs={"imgsz": 640, "batch": 16, "workers": 0},
    predict_kwargs={"conf": 0.25},
    lightweight_return=False
)
cell_elapsed_seconds = time.perf_counter() - cell_start

print(f"Czas wykonania komórki: {cell_elapsed_seconds:.2f} s")
mitl_result["statistics"] if mitl_result is not None else None

Initial training: 20 epochs on 1793 images
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\rwcki\AppData\Local\Temp\model_in_loop_el5l14jb\data_full.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=

2026/07/21 10:02:57 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/07/21 10:02:57 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/07/21 10:02:57 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(0a71f8bd02bd4158a8cdc29efaa1b1d8) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\Politechnika\Deadlift_publikacja_nr_2  kopia\Skrypty\github\scores_from_publication\runs\pose\train5
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
       1/20      2.31G      1.943      8.062     0.7107      2.295      1.772          2        640: 100% ━━━━━━━━━━━━ 113/113 1.8it/s 1:030.4ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
       2/20      2.31G      1.723      5.487     0.5703      1.507      1.552          2        640: 100% ━━━━━━━━━━━━ 113/113 1.9it/s 1:000.4ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss 

Predicting keypoints (after_initial_training): 100%|██████████| 1793/1793 [01:05<00:00, 27.25it/s]

Continuing training: 20 epochs on 1434 images (359 removed)
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)


...

2026/07/21 10:22:33 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/07/21 10:22:33 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/07/21 10:22:33 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(6f8fc853c378408a88e1360187bdbfda) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\Politechnika\Deadlift_publikacja_nr_2  kopia\Skrypty\github\scores_from_publication\runs\pose\train6
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
       1/20      2.37G      1.652      2.963      0.405      1.514      1.585         21        640: 100% ━━━━━━━━━━━━ 90/90 2.0it/s 44.8s0.4ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
       2/20      2.38G      1.244      2.061     0.3899     0.9986      1.248         20        640: 100% ━━━━━━━━━━━━ 90/90 2.1it/s 43.1s0.5ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  I

Predicting keypoints (after_reduced_training): 100%|██████████| 1793/1793 [01:06<00:00, 27.11it/s]

Czas wykonania komórki: 2100.57 s


{'initial_epochs': 20,
 'fine_tune_epochs': 20,
 'remove_percent': 20.0,
 'initial_train_image_count': 1793,
 'reduced_train_image_count': 1434,
 'removed_image_count': 359,
 'removed_record_count': 359,
 'final_total_records': 1919,
 'work_dir': 'C:\\Users\\rwcki\\AppData\\Local\\Temp\\model_in_loop_el5l14jb'}

In [8]:
# Porownanie wynikow MITL z trzema plikami referencyjnymi (ile wykrylismy z tych, ktore powinnismy)
reference_report_paths = {
    "angle_preserving": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1_angle_preserving_out\results.txt",
    "rigid": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2_rigid_out\results.txt",
    "axis_aligned": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3_axis_out\results.txt",
}

mitl_evaluator = MergedRankingEvaluator(
    validator=validator,
    report_paths=reference_report_paths,
    top_k=450,
    output_path=None,
)

reference_entries_map = mitl_evaluator._load_reference_report_entries()
mitl_comparisons = {}
for report_name, reference_entries in reference_entries_map.items():
    mitl_comparisons[report_name] = mitl_evaluator._compare_reference_entries_to_anomalies(
        reference_entries,
        mitl_result["combined_ranking"],
    )

mitl_comparisons

{'angle_preserving': {'matched_top_k_count': 54,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 561.0,
  'last_matched_rank_in_top_k': 450},
 'rigid': {'matched_top_k_count': 144,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 179.68666666666667,
  'last_matched_rank_in_top_k': 431},
 'axis_aligned': {'matched_top_k_count': 133,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 300.66,
  'last_matched_rank_in_top_k': 398},
 'merged': {'matched_top_k_count': 331,
  'total_reference_entries': 450,
  'matched_overall_count': 450,
  'avg_rank_all_reference': 347.11555555555555,
  'last_matched_rank_in_top_k': 450}}

In [9]:
# MITL (Model-In-The-Loop) - wywolanie test_mitl
cell_start = time.perf_counter()
mitl_result = test_mitl(
    x=450,
    model="yolo11n-pose.pt",
    initial_epochs=5,
    remove_percent=20,
    fine_tune_epochs=5,
    train_kwargs={"imgsz": 640, "batch": 16, "workers": 0},
    predict_kwargs={"conf": 0.25},
    lightweight_return=False
)
cell_elapsed_seconds = time.perf_counter() - cell_start

print(f"Czas wykonania komórki: {cell_elapsed_seconds:.2f} s")
mitl_result["statistics"] if mitl_result is not None else None

Initial training: 5 epochs on 1793 images
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\rwcki\AppData\Local\Temp\model_in_loop_r53khocu\data_full.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tr

2026/07/21 11:38:18 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/07/21 11:38:18 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/07/21 11:38:18 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(10e2479affab431e8b82aed4ce7e70ef) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\Politechnika\Deadlift_publikacja_nr_2  kopia\Skrypty\github\scores_from_publication\runs\pose\train7
Starting training for 5 epochs...

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
        1/5      2.37G      1.943      8.062     0.7107      2.295      1.772          2        640: 100% ━━━━━━━━━━━━ 113/113 1.6it/s 1:100.4ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
        2/5      2.37G      1.678      5.457      0.579      1.443      1.523          2        640: 100% ━━━━━━━━━━━━ 113/113 1.6it/s 1:100.4ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  

Predicting keypoints (after_initial_training): 100%|██████████| 1793/1793 [01:28<00:00, 20.23it/s]

Continuing training: 5 epochs on 1434 images (359 removed)
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\rwcki\AppData\Local\Temp\model_in_loop_r53khocu\data_reduced.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, mult

...

2026/07/21 11:46:45 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/07/21 11:46:45 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/07/21 11:46:45 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(588b78c6b73a436ab13cad95e28c032f) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\Politechnika\Deadlift_publikacja_nr_2  kopia\Skrypty\github\scores_from_publication\runs\pose\train8
Starting training for 5 epochs...

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
        1/5      2.37G      1.823      3.568     0.4365       1.74      1.718         17        640: 100% ━━━━━━━━━━━━ 90/90 1.6it/s 57.4s0.5ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
        2/5      2.38G       1.37      2.503     0.4324      1.121      1.319         15        640: 100% ━━━━━━━━━━━━ 90/90 1.7it/s 54.5s0.6ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  In

Predicting keypoints (after_reduced_training): 100%|██████████| 1793/1793 [01:27<00:00, 20.53it/s]

Czas wykonania komórki: 930.47 s


{'initial_epochs': 5,
 'fine_tune_epochs': 5,
 'remove_percent': 20.0,
 'initial_train_image_count': 1793,
 'reduced_train_image_count': 1434,
 'removed_image_count': 359,
 'removed_record_count': 359,
 'final_total_records': 1919,
 'work_dir': 'C:\\Users\\rwcki\\AppData\\Local\\Temp\\model_in_loop_r53khocu'}

In [11]:
# Porownanie wynikow MITL z trzema plikami referencyjnymi (ile wykrylismy z tych, ktore powinnismy)
reference_report_paths = {
    "angle_preserving": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1_angle_preserving_out\results.txt",
    "rigid": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2_rigid_out\results.txt",
    "axis_aligned": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3_axis_out\results.txt",
}

mitl_evaluator = MergedRankingEvaluator(
    validator=validator,
    report_paths=reference_report_paths,
    top_k=450,
    output_path=None,
)

reference_entries_map = mitl_evaluator._load_reference_report_entries()
mitl_comparisons = {}
for report_name, reference_entries in reference_entries_map.items():
    mitl_comparisons[report_name] = mitl_evaluator._compare_reference_entries_to_anomalies(
        reference_entries,
        mitl_result["combined_ranking"],
    )

mitl_comparisons

{'angle_preserving': {'matched_top_k_count': 34,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 663.9866666666667,
  'last_matched_rank_in_top_k': 447},
 'rigid': {'matched_top_k_count': 139,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 227.3,
  'last_matched_rank_in_top_k': 430},
 'axis_aligned': {'matched_top_k_count': 128,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 350.6066666666667,
  'last_matched_rank_in_top_k': 448},
 'merged': {'matched_top_k_count': 301,
  'total_reference_entries': 450,
  'matched_overall_count': 450,
  'avg_rank_all_reference': 413.96444444444444,
  'last_matched_rank_in_top_k': 448}}

In [12]:
# MITL (Model-In-The-Loop) - wywolanie test_mitl
cell_start = time.perf_counter()
mitl_result = test_mitl(
    x=450,
    model="yolo11n-pose.pt",
    initial_epochs=2,
    remove_percent=20,
    fine_tune_epochs=3,
    train_kwargs={"imgsz": 640, "batch": 16, "workers": 0},
    predict_kwargs={"conf": 0.25},
    lightweight_return=False
)
cell_elapsed_seconds = time.perf_counter() - cell_start

print(f"Czas wykonania komórki: {cell_elapsed_seconds:.2f} s")
mitl_result["statistics"] if mitl_result is not None else None

Initial training: 2 epochs on 1793 images
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\rwcki\AppData\Local\Temp\model_in_loop_8arfzk0j\data_full.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tr

2026/07/21 11:54:38 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/07/21 11:54:38 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/07/21 11:54:38 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(fdadb41de5f84e02b4369bb409ae3e45) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\Politechnika\Deadlift_publikacja_nr_2  kopia\Skrypty\github\scores_from_publication\runs\pose\train9
Starting training for 2 epochs...

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
        1/2      2.38G      1.943      8.062     0.7107      2.295      1.772          2        640: 100% ━━━━━━━━━━━━ 113/113 1.6it/s 1:110.5ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
        2/2      2.38G      1.591      5.571     0.6073       1.37      1.453          2        640: 100% ━━━━━━━━━━━━ 113/113 1.7it/s 1:080.4ss
                 Class     Images  Instances      Box(P          R      mAP50  m

Predicting keypoints (after_initial_training): 100%|██████████| 1793/1793 [01:26<00:00, 20.68it/s]

Continuing training: 3 epochs on 1434 images (359 removed)
Ultralytics 8.4.14  Python-3.10.19 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)


...

2026/07/21 11:59:37 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/07/21 11:59:37 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/07/21 11:59:37 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(38ec483fd48a44a29a9a30870e3edde4) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\Politechnika\Deadlift_publikacja_nr_2  kopia\Skrypty\github\scores_from_publication\runs\pose\train10
Starting training for 3 epochs...

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
        1/3      2.36G      2.033      4.998     0.5121      1.943      1.824         20        640: 100% ━━━━━━━━━━━━ 90/90 1.6it/s 56.3s0.6ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
        2/3      2.37G      1.486      3.614     0.4517      1.277      1.362         14        640: 100% ━━━━━━━━━━━━ 90/90 1.6it/s 55.0s0.6ss

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  I

Predicting keypoints (after_reduced_training): 100%|██████████| 1793/1793 [01:28<00:00, 20.19it/s]

Czas wykonania komórki: 615.60 s


{'initial_epochs': 2,
 'fine_tune_epochs': 3,
 'remove_percent': 20.0,
 'initial_train_image_count': 1793,
 'reduced_train_image_count': 1434,
 'removed_image_count': 359,
 'removed_record_count': 359,
 'final_total_records': 1919,
 'work_dir': 'C:\\Users\\rwcki\\AppData\\Local\\Temp\\model_in_loop_8arfzk0j'}

In [13]:
# Porownanie wynikow MITL z trzema plikami referencyjnymi (ile wykrylismy z tych, ktore powinnismy)
reference_report_paths = {
    "angle_preserving": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_1_angle_preserving_out\results.txt",
    "rigid": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_2_rigid_out\results.txt",
    "axis_aligned": r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\part_3_axis_out\results.txt",
}

mitl_evaluator = MergedRankingEvaluator(
    validator=validator,
    report_paths=reference_report_paths,
    top_k=450,
    output_path=None,
)

reference_entries_map = mitl_evaluator._load_reference_report_entries()
mitl_comparisons = {}
for report_name, reference_entries in reference_entries_map.items():
    mitl_comparisons[report_name] = mitl_evaluator._compare_reference_entries_to_anomalies(
        reference_entries,
        mitl_result["combined_ranking"],
    )

mitl_comparisons

{'angle_preserving': {'matched_top_k_count': 20,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 872.84,
  'last_matched_rank_in_top_k': 435},
 'rigid': {'matched_top_k_count': 126,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 327.46666666666664,
  'last_matched_rank_in_top_k': 447},
 'axis_aligned': {'matched_top_k_count': 94,
  'total_reference_entries': 150,
  'matched_overall_count': 150,
  'avg_rank_all_reference': 463.76,
  'last_matched_rank_in_top_k': 446},
 'merged': {'matched_top_k_count': 240,
  'total_reference_entries': 450,
  'matched_overall_count': 450,
  'avg_rank_all_reference': 554.6888888888889,
  'last_matched_rank_in_top_k': 447}}

## Replacement of files which were not detected 

In [4]:
from pathlib import Path
import os
import shutil

from utils.validation import parse_results_txt, YPValidation_Test


def _collect_image_paths(dataset_root: str):
    dataset_root_path = Path(dataset_root)
    images_root = dataset_root_path / "images"
    if not images_root.exists():
        raise FileNotFoundError(f"Brak folderu images: {images_root}")

    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return [p for p in images_root.rglob("*") if p.is_file() and p.suffix.lower() in image_exts]


def _image_relative_key(image_path: Path):
    normalized = str(image_path).replace('\\', '/')
    marker = '/images/'
    idx = normalized.lower().find(marker)
    if idx == -1:
        return None
    return normalized[idx + 1 :]


def _build_indexes(dataset_root: str):
    by_relative = {}
    by_basename = {}

    for img_path in _collect_image_paths(dataset_root):
        rel_key = _image_relative_key(img_path)
        if rel_key is not None:
            by_relative[rel_key.lower()] = img_path

        basename = img_path.name.lower()
        by_basename.setdefault(basename, []).append(img_path)

    return by_relative, by_basename


def _resolve_image(entry_image_path: str, by_relative: dict, by_basename: dict):
    if entry_image_path:
        rel_key = _image_relative_key(Path(entry_image_path))
        if rel_key is not None:
            hit = by_relative.get(rel_key.lower())
            if hit is not None:
                return hit

        basename = Path(entry_image_path).name.lower()
        candidates = by_basename.get(basename, [])
        if len(candidates) == 1:
            return candidates[0]

    return None


def _label_path_from_image(image_path: Path):
    parts = list(image_path.parts)
    lowered = [p.lower() for p in parts]
    if "images" in lowered:
        i = lowered.index("images")
        label_parts = parts[:]
        label_parts[i] = "labels"
        label_path = Path(*label_parts).with_suffix('.txt')
        return label_path

    return image_path.parent.parent / "labels" / f"{image_path.stem}.txt"


def _identifier(entry):
    return YPValidation_Test._distance_anomaly_entry_identifier(entry, "image_path_person_id")


def _undetected_from_combined_ranking(payload, reference_results_txt_path: str, top_k: int):
    combined_ranking = payload.get("combined_ranking", []) if isinstance(payload, dict) else []
    if not combined_ranking:
        return []

    reference_entries = parse_results_txt(reference_results_txt_path)
    ranked_entries = sorted(combined_ranking, key=lambda x: x.get("rank", float("inf")))
    top_entries = [entry for entry in ranked_entries if int(entry.get("rank", 10**9)) <= int(top_k)]

    top_buckets = {}
    for entry in top_entries:
        key = _identifier(entry)
        top_buckets.setdefault(key, []).append(entry)

    undetected = []
    for reference_entry in reference_entries:
        key = _identifier(reference_entry)
        bucket = top_buckets.get(key, [])
        if bucket:
            bucket.pop(0)
        else:
            undetected.append(reference_entry)

    return undetected


def _extract_undetected_entries(
    report_payload: dict,
    report_name: str,
    reference_results_txt_path: str = None,
    top_k: int = None,
):
    if report_payload is None:
        return [], "Brak payloadu z wynikiem"

    # Tryb 1: gotowa lista brakow jest juz w payloadzie.
    if report_name in report_payload and isinstance(report_payload[report_name], dict):
        undetected_entries = report_payload[report_name].get("undetected_entries", [])
        if undetected_entries:
            return undetected_entries, None

    if "undetected_entries" in report_payload:
        undetected_entries = report_payload.get("undetected_entries", [])
        if undetected_entries:
            return undetected_entries, None

    # Tryb 2: payload to pelny wynik evaluator.run() i rekonstruujemy braki z top_k.
    if reference_results_txt_path and top_k is not None and isinstance(report_payload, dict):
        if "combined_ranking" in report_payload:
            return _undetected_from_combined_ranking(report_payload, reference_results_txt_path, top_k), None

    warning = (
        "Nie znaleziono undetected_entries. "
        "Uzyj pelnego wyniku z test() (z combined_ranking), podaj reference_results_txt_path i top_k, "
        "a potem uruchom komorke ponownie."
    )
    return [], warning


def replace_undetected_topk_files_in_clean_dataset(
    report_payload: dict,
    dirty_dataset_root: str,
    clean_dataset_root: str,
    report_name: str = "merged",
    copy_labels: bool = False,
    dry_run: bool = False,
    reference_results_txt_path: str = None,
    top_k: int = None,
):
    """
    Nadpisuje tylko obrazy nieznalezione w top_k (undetected_entries) w dataset_czysty,
    obrazami z dataset_brudny.

    Parametry:
    - report_payload: wynik z test(...), najlepiej pelny (z combined_ranking)
    - dirty_dataset_root: root dataset_brudny
    - clean_dataset_root: root dataset_czysty_bez_duplikatow
    - report_name: domyslnie "merged"
    - copy_labels: jezeli True, nadpisuje tez odpowiadajace pliki label .txt
    - dry_run: jezeli True, nic nie kopiuje, tylko zwraca plan podmian
    - reference_results_txt_path: np. ...\partitions\merged_results.txt (wymagane przy rekonstrukcji brakow)
    - top_k: np. 450 (wymagane przy rekonstrukcji brakow)
    """
    undetected_entries, warning = _extract_undetected_entries(
        report_payload=report_payload,
        report_name=report_name,
        reference_results_txt_path=reference_results_txt_path,
        top_k=top_k,
    )

    if not undetected_entries:
        return {
            "undetected_total": 0,
            "planned_replacements_count": 0,
            "planned_replacements": [],
            "replaced_count": 0,
            "replaced_images": [],
            "skipped_missing_in_dirty": [],
            "skipped_missing_in_clean": [],
            "skipped_ambiguous": [],
            "copied_labels_count": 0,
            "dry_run": dry_run,
            "warning": warning,
        }

    dirty_by_rel, dirty_by_name = _build_indexes(dirty_dataset_root)
    clean_by_rel, clean_by_name = _build_indexes(clean_dataset_root)

    unique_image_paths = []
    seen = set()
    for entry in undetected_entries:
        image_path = str(entry.get("image_path", "")).strip()
        if not image_path:
            continue
        key = image_path.replace('\\', '/').lower()
        if key not in seen:
            seen.add(key)
            unique_image_paths.append(image_path)

    planned_replacements = []
    replaced_images = []
    skipped_missing_in_dirty = []
    skipped_missing_in_clean = []
    skipped_ambiguous = []
    copied_labels_count = 0

    for image_path in unique_image_paths:
        source_img = _resolve_image(image_path, dirty_by_rel, dirty_by_name)
        target_img = _resolve_image(image_path, clean_by_rel, clean_by_name)

        if source_img is None:
            name = Path(image_path).name.lower()
            if len(dirty_by_name.get(name, [])) > 1:
                skipped_ambiguous.append({"dataset": "dirty", "image_path": image_path})
            else:
                skipped_missing_in_dirty.append(image_path)
            continue

        if target_img is None:
            name = Path(image_path).name.lower()
            if len(clean_by_name.get(name, [])) > 1:
                skipped_ambiguous.append({"dataset": "clean", "image_path": image_path})
            else:
                skipped_missing_in_clean.append(image_path)
            continue

        plan_item = {
            "source_image": str(source_img),
            "target_image": str(target_img),
        }

        if copy_labels:
            source_lbl = _label_path_from_image(source_img)
            target_lbl = _label_path_from_image(target_img)
            plan_item["source_label"] = str(source_lbl)
            plan_item["target_label"] = str(target_lbl)

        planned_replacements.append(plan_item)

        if dry_run:
            continue

        shutil.copy2(source_img, target_img)
        replaced_images.append(str(target_img))

        if copy_labels:
            source_lbl = _label_path_from_image(source_img)
            target_lbl = _label_path_from_image(target_img)
            if source_lbl.exists() and target_lbl.exists():
                shutil.copy2(source_lbl, target_lbl)
                copied_labels_count += 1

    return {
        "undetected_total": len(unique_image_paths),
        "planned_replacements_count": len(planned_replacements),
        "planned_replacements": planned_replacements,
        "replaced_count": len(replaced_images),
        "replaced_images": replaced_images,
        "skipped_missing_in_dirty": skipped_missing_in_dirty,
        "skipped_missing_in_clean": skipped_missing_in_clean,
        "skipped_ambiguous": skipped_ambiguous,
        "copied_labels_count": copied_labels_count,
        "dry_run": dry_run,
        "warning": warning,
    }

In [5]:
replacement_preview = replace_undetected_topk_files_in_clean_dataset(
    report_payload=merged_450,
    dirty_dataset_root=r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset_brudny",
    clean_dataset_root=r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\full_czysty_dataset",
    report_name="merged",
    copy_labels=True,
    dry_run=False,
    reference_results_txt_path=r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\partitions\merged_results.txt",
    top_k=450,
)

if replacement_preview.get("warning"):
    print("warning:", replacement_preview["warning"])

print("undetected_total:", replacement_preview["undetected_total"])
print("planned_replacements_count:", replacement_preview["planned_replacements_count"])
print("skipped_missing_in_dirty:", len(replacement_preview["skipped_missing_in_dirty"]))
print("skipped_missing_in_clean:", len(replacement_preview["skipped_missing_in_clean"]))
print("skipped_ambiguous:", len(replacement_preview["skipped_ambiguous"]))

replacement_preview["planned_replacements"][:10]

undetected_total: 131
planned_replacements_count: 131
skipped_missing_in_dirty: 0
skipped_missing_in_clean: 0
skipped_ambiguous: 0


[{'source_image': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\final_augmented_dataset_brudny\\images\\deadlift-side-view_frame_23_jpg.rf.353fb561b80178f11c57ace140582c2b.jpg',
  'target_image': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\full_czysty_dataset\\images\\deadlift-side-view_frame_23_jpg.rf.353fb561b80178f11c57ace140582c2b.jpg',
  'source_label': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\final_augmented_dataset_brudny\\labels\\deadlift-side-view_frame_23_jpg.rf.353fb561b80178f11c57ace140582c2b.txt',
  'target_label': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\full_czysty_dataset\\labels\\deadlift-side-view_frame_23_jpg.rf.353fb561b80178f11c57ace140582c2b.txt'},
 {'source_image': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\final_augmented_dataset_brudny\\images\\050741194.jpg',
  'target_image': 'D:\\Politechnika\\Deadlift_publikacja_nr_2\\Dane_do_treningu\\full_czysty_dataset\\im

## SPLITTING DATASETS BEFORE TRAINING

In [ ]:
split_yolo_pose_dataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset_brudny",    r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\final_augmented_dataset_brudny_split", val_ratio=0.1, copy_mode= True)

In [6]:
split_yolo_pose_dataset(r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\full_czysty_dataset",    r"D:\Politechnika\Deadlift_publikacja_nr_2\Dane_do_treningu\full_czysty_dataset_bez_duplikatow_split_2", val_ratio=0.1, copy_mode = True)